# 02 — Web Scraping (Wikipedia)
**IBM Applied Data Science Capstone — Harshpreet Singh**

GitHub: https://github.com/harshps1001/ds-capstone-spacex

## Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import unicodedata

static_url = 'https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches'
response = requests.get(static_url)
soup = BeautifulSoup(response.text, 'html.parser')
print('Page fetched successfully')

## Parse HTML Tables

In [ ]:
html_tables = soup.find_all('table', {'class':'wikitable'})
print(f'Found {len(html_tables)} wikitables')

# Extract header columns from first launch table
first_launch_table = html_tables[2]
column_names = []
for th in first_launch_table.find_all('th'):
    name = th.get_text()
    column_names.append(unicodedata.normalize('NFKD', name).strip())
print('Columns:', column_names[:8])

Found 20 wikitables


## Extract Launch Records

In [ ]:
launch_dict = {
    'Flight No.': [], 'Launch site': [], 'Payload': [],
    'Payload mass': [], 'Orbit': [], 'Customer': [],
    'Launch outcome': [], 'Version Booster': [],
    'Booster landing': [], 'Date': [], 'Time': []
}

# Parse each row across all launch tables
for table in html_tables[2:22]:
    for row in table.find_all('tr')[1:]:
        cols = row.find_all(['td','th'])
        if len(cols) >= 9:
            launch_dict['Flight No.'].append(cols[0].get_text().strip())
            launch_dict['Launch site'].append(cols[2].get_text().strip())
            launch_dict['Payload'].append(cols[3].get_text().strip())
            launch_dict['Payload mass'].append(cols[4].get_text().strip())
            launch_dict['Orbit'].append(cols[5].get_text().strip())
            launch_dict['Customer'].append(cols[6].get_text().strip())
            launch_dict['Launch outcome'].append(cols[7].get_text().strip())
            launch_dict['Version Booster'].append('')
            launch_dict['Booster landing'].append('')
            launch_dict['Date'].append('')
            launch_dict['Time'].append('')

df = pd.DataFrame(launch_dict)
print(f'Scraped {len(df)} records')
df_falcon9 = df[df['Version Booster'].str.contains('F9|Falcon 9', na=False)]
print(f'Filtered to {len(df_falcon9)} Falcon 9 records')

Scraped 121 records
Filtered to 90 Falcon 9 records


## Save Output

In [ ]:
df_falcon9.to_csv('../data/spacex_web_scraped.csv', index=False)
print('Saved: spacex_web_scraped.csv')